In [9]:
import tensorflow as tf
import numpy as np
import librosa
import os
import pandas as pd

In [11]:
# --- CONFIGURATION & CONSTANTS ---
# These must be the same as the values used during training.
TARGET_SR = 16000
N_MFCC = 13
HOP_LENGTH = 512
N_FFT = 2048
MAX_FRAMES = 200

In [52]:
# --- LOAD THE TRAINED MODEL AND CLASS LABELS ---
MODEL_PATH = '/home/sadgeu/projects/CNN-BiGRU/best_model_BiGRU_real.keras'
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model file not found at '{MODEL_PATH}'. "
        "Please make sure the trained model file is in the same directory as this script."
    )

# Load the trained Keras model
print(f"Loading trained model from: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully.")

# Define the class labels in the same order as the training process.
# This order is derived from the classification report you provided.
CLASS_LABELS = [
    'Bronchiectasis', 'Bronchiolitis', 'COPD', 'Healthy', 'Pneumonia', 'URTI'
]

Loading trained model from: /home/sadgeu/projects/CNN-BiGRU/best_model_BiGRU_real.keras
Model loaded successfully.


In [13]:
# --- FEATURE EXTRACTION & PRE-PROCESSING ---

def preprocess_single_audio(wav_path):
    """
    Loads a single audio file and applies the pre-processing pipeline.
    This includes feature extraction and formatting.
    """
    try:
        # 1. Load audio file and resample to target sample rate
        wav, sr = librosa.load(wav_path, sr=TARGET_SR)
        
        # 2. Extract features (MFCCs + Deltas)
        mfcc = librosa.feature.mfcc(y=wav, sr=TARGET_SR, n_mfcc=N_MFCC, hop_length=HOP_LENGTH, n_fft=N_FFT)
        mfcc_delta = librosa.feature.delta(mfcc)
        mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
        features = np.vstack([mfcc, mfcc_delta, mfcc_delta2]) # Shape: (39, T)

        # 3. Pad or truncate to MAX_FRAMES
        D, T = features.shape
        if T < MAX_FRAMES:
            pad_width = MAX_FRAMES - T
            features = np.pad(features, ((0, 0), (0, pad_width)), mode="constant")
        elif T > MAX_FRAMES:
            features = features[:, :MAX_FRAMES]
        
        # 4. Transpose to match model input shape (T, D)
        features = features.T # Shape: (200, 39)

        # 5. Add batch dimension for the model
        features = np.expand_dims(features, axis=0) # Shape: (1, 200, 39)

        return features

    except Exception as e:
        print(f"Error processing file {wav_path}: {e}")
        return None


In [14]:
def predict_from_file(audio_path):
    """
    Takes an audio file path, preprocesses it, and returns the predicted diagnosis.
    """
    # Pre-process the audio file to get the feature vector
    processed_features = preprocess_single_audio(audio_path)

    if processed_features is None:
        return "Error in processing", 0.0

    # Make a prediction using the loaded model
    prediction_probabilities = model.predict(processed_features)[0]

    # Find the index of the class with the highest probability
    predicted_index = np.argmax(prediction_probabilities)
    
    # Get the corresponding class label
    predicted_label = CLASS_LABELS[predicted_index]
    
    # Get the confidence score of the prediction
    confidence = prediction_probabilities[predicted_index]

    return predicted_label, confidence



In [17]:
# --- NEW PREDICTION & VERIFICATION FUNCTION ---

def predict_and_verify(audio_path, metadata_df):
    """
    Takes an audio file path and a metadata DataFrame, makes a prediction,
    and verifies it against the ground truth label by looking up the Patient ID.
    """
    # Pre-process the audio file to get the feature vector
    processed_features = preprocess_single_audio(audio_path)

    if processed_features is None:
        return "Error in processing", 0.0, "N/A", "Error"

    # --- Prediction Step ---
    prediction_probabilities = model.predict(processed_features, verbose=0)[0]
    predicted_index = np.argmax(prediction_probabilities)
    predicted_label = CLASS_LABELS[predicted_index]
    confidence = prediction_probabilities[predicted_index]

    # --- Verification Step ---
    ground_truth_label = "Label not found"
    is_correct = "Cannot Verify"
    
    try:
        # Extract patient ID from the filename (e.g., "101_1b1_Al_sc_Meditron.wav" -> 101)
        filename = os.path.basename(audio_path)
        patient_id = int(filename.split('_')[0])

        # Find the row corresponding to the patient_id
        # Assumes the first column of the CSV is the patient ID (column index 0)
        ground_truth_row = metadata_df[metadata_df[0] == patient_id]

        if not ground_truth_row.empty:
            # Get the true diagnosis from the second column (column index 1)
            ground_truth_label = ground_truth_row.iloc[0][1]
            is_correct = "CORRECT" if predicted_label == ground_truth_label else "INCORRECT"

    except (ValueError, IndexError, KeyError) as e:
        # Handle cases where filename format is unexpected or CSV is wrong
        print(f"Could not parse patient ID or find label for '{filename}'. Error: {e}")
        ground_truth_label = "Could not parse/find label"
        is_correct = "Cannot Verify"

    return predicted_label, confidence, ground_truth_label, is_correct



In [18]:
# Load your test metadata DataFrame, specifying it has no header
# This will create columns named 0, 1, etc.
metadata_file_path = '/home/sadgeu/projects/CNN-BiGRU/Respiratory_Sound_Database/Respiratory_Sound_Database/patient_diagnosis.csv'
test_df = pd.read_csv(metadata_file_path, header=None)

In [51]:
# Path to the audio file from your test set you want to use as an example
audio_file_to_test = '/home/sadgeu/projects/CNN-BiGRU/inference_test_wavs/197_1b1_Tc_sc_Meditron.wav'

# Call the new function, passing the audio path AND your test DataFrame
predicted_diagnosis, confidence_score, true_diagnosis, result_status = predict_and_verify(
    audio_file_to_test, 
    test_df 
)

# Print the detailed result
print("\n--- INFERENCE & VERIFICATION ---")
print(f"File Analyzed:       {os.path.basename(audio_file_to_test)}")
print("-" * 30)
print(f"Model Prediction:    {predicted_diagnosis}")
print(f"Confidence:          {confidence_score:.2%}")
print(f"Actual Diagnosis:    {true_diagnosis}")
print("-" * 30)
print(f"Result:              {result_status}")
print("--------------------------------\n")


--- INFERENCE & VERIFICATION ---
File Analyzed:       197_1b1_Tc_sc_Meditron.wav
------------------------------
Model Prediction:    URTI
Confidence:          98.22%
Actual Diagnosis:    URTI
------------------------------
Result:              CORRECT
--------------------------------

